####Paso 1 - Leer el archivo JSON usando "DataFrameReader de Spark"

In [0]:
%run "../includes/configurations"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
movies_languages_schema = StructType(fields=[
  StructField("movieId", IntegerType(), True),
  StructField("languageId", IntegerType(), True),
  StructField("languageRoleId", IntegerType(), True)
])

In [0]:
movies_languages_df = spark.read \
  .schema(movies_languages_schema) \
  .option("multiline", True) \
  .json(f"{bronze_folder_path}/{v_file_date}/movie_language")

In [0]:
display(movies_languages_df)

####Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_languages_with_columns_df = movies_languages_df \
    .withColumnsRenamed({"movieId": "movie_id",
                        "languageId": "language_id"}) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("enviroment", lit("Produccion")) \
    .withColumn("file_date", lit(v_file_date))

In [0]:
display(movies_languages_with_columns_df)

####Paso 3 - Eliminar las columnas no deseadas del DataFrame

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_languages_final_df = movies_languages_with_columns_df.drop("languageRoleId")

In [0]:
display(movies_languages_final_df)

####Paso 4 - Escribir la salida en formato "Parquet"

In [0]:
#overwrite_partition("movie_silver", "movies_languages", "file_date", v_file_date)

In [0]:
#movies_languages_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/movie_language")
#movies_languages_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_languages")
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.file_date = src.file_date'
merge_delta_lake (movies_languages_final_df, "movie_silver", "movies_languages", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies_languages
GROUP BY file_date;

In [0]:
%sql
SELECT *
FROM movie_silver.movies_languages;

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/movie_language

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/movie_language"))